# Cell 1: Project Overview

# Temporal Reference-Guided Diffusion Network (TRDN) for Video Dehazing

This notebook is the **primary execution interface** for the repository. All research logic lives in `src/`; this notebook only performs setup, configuration, visualization, debugging, and launch control for training/validation/inference.

Execution target: **VS Code notebook + Google Colab runtime**.

Core pipeline:

`REVIDE sequence -> RAFT/warping -> ConvLSTM -> Temporal Retrieval Transformer -> Reference Selection -> Temporal Conditioning Adapter -> Stable Diffusion Inpainting UNet`.


In [ ]:
# Cell 2: Install Dependencies
# Repository-first rule: install packages only; do not define model/dataset logic here.
import sys
import subprocess
from pathlib import Path

requirements_path = Path("requirements_colab.txt")
if requirements_path.exists():
    cmd = [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)]
else:
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "accelerate>=0.31.0", "diffusers>=0.30.0", "transformers>=4.41.0",
        "peft>=0.11.0", "tensorboard>=2.16.0", "lpips>=0.1.4",
        "torchmetrics>=1.4.0", "opencv-python>=4.8.0", "scikit-image>=0.22.0",
        "matplotlib>=3.7.0", "tqdm>=4.66.0"
    ]
print("Installing/checking dependencies...")
subprocess.check_call(cmd)
print("Dependency setup complete.")


In [ ]:
# Cell 3: Mount Google Drive
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    COLAB = True
except Exception as exc:
    COLAB = False
    print("Google Drive mount skipped or unavailable:", repr(exc))

DRIVE_ROOT = Path("/content/drive/MyDrive") if COLAB else Path.cwd()
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("COLAB:", COLAB)
print("DRIVE_ROOT:", DRIVE_ROOT)


In [ ]:
# Cell 4: Clone/Open Repository
# If this notebook is opened from the repo in VS Code, this cell simply locates it.
# If running directly in Colab and src/ is absent, it attempts to clone the repo.
import os
import sys
import subprocess
from pathlib import Path

REPO_NAME = "TRDN-Video-Dehazing"
REPO_URL = "https://github.com/amir1373/TRDN-Video-Dehazing.git"

candidates = [
    Path.cwd(),
    Path.cwd().parent,
    DRIVE_ROOT / REPO_NAME,
    Path("/content") / REPO_NAME,
]
REPO_ROOT = next((p.resolve() for p in candidates if (p / "src").exists()), None)

if REPO_ROOT is None:
    clone_dir = Path("/content") / REPO_NAME if COLAB else Path.cwd() / REPO_NAME
    token = os.environ.get("GITHUB_TOKEN", "").strip()
    clone_url = REPO_URL
    if token and REPO_URL.startswith("https://github.com/"):
        clone_url = REPO_URL.replace("https://github.com/", f"https://{token}@github.com/")
    print(f"src/ not found. Cloning repository into {clone_dir}...")
    result = subprocess.run(["git", "clone", clone_url, str(clone_dir)], text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(
            "Could not clone/open repository. If the GitHub repo is private, set the GITHUB_TOKEN "
            "environment variable in Colab or open this notebook from a checked-out repo in VS Code.\n"
            f"git stderr:\n{result.stderr}"
        )
    REPO_ROOT = clone_dir.resolve()

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("REPO_ROOT:", REPO_ROOT)
print("src exists:", (REPO_ROOT / "src").exists())


In [ ]:
# Cell 5: Set Paths
# All storage paths are derived from DRIVE_ROOT and src.config.TRDNConfig.
import json
import torch
from pathlib import Path
from src.config import TRDNConfig

cfg = TRDNConfig()
cfg.project_root = str(DRIVE_ROOT / "TRDN_REVIDE")

# Central REVIDE paths. Edit here only if your Drive layout differs.
cfg.train_root = "/content/drive/MyDrive/REVIDE_sequences/Train"
cfg.test_root = "/content/drive/MyDrive/REVIDE_sequences/Test"
cfg.train_hazy = "/content/drive/MyDrive/REVIDE_sequences/Train/hazy"
cfg.test_hazy = "/content/drive/MyDrive/REVIDE_sequences/Test/hazy"
cfg.flow_train = "/content/drive/MyDrive/video_dehaze_flows/train"
cfg.flow_val = "/content/drive/MyDrive/video_dehaze_flows/val"
cfg.dataset_root = "/content/drive/MyDrive/REVIDE_sequences"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PATHS = cfg.ensure_dirs()

print("DEVICE:", DEVICE)
print(json.dumps(cfg.to_dict(), indent=2, default=str))
print("Output directories:")
for name, path in PATHS.items():
    print(f"  {name}: {path}")


In [ ]:
# Cell 6: Imports from src/
# This cell imports repository modules. It intentionally contains no core implementations.
import json
import math
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from torchvision.utils import save_image
from accelerate.utils import set_seed

from src.dataset import REVIDESequenceDataset, discover_revide_sequences
from src.masks import generate_haze_mask
from src.haze import simulate_realistic_haze
from src.flow import compute_warped_references_batch, flow_to_rgb, load_raft
from src.convlstm import TemporalMemoryModule
from src.temporal_transformer import TemporalRetrievalTransformer
from src.reference_selector import ReferenceSelectionModule
from src.diffusion_adapter import TemporalConditioningAdapter, load_diffusion_backbone
from src.losses import LossBundle
from src.metrics import psnr_metric, ssim_metric
from src.train import build_temporal_modules, dry_run_shape_test, train_trdn
from src.validate import infer_dehazed_batch, validate_trdn
from src.inference import run_inference_on_index

set_seed(cfg.seed)

def show_tensor_images(images, titles, figsize=(16, 4), save_path=None):
    plt.figure(figsize=figsize)
    for idx, (image, title) in enumerate(zip(images, titles), start=1):
        x = image.detach().float().cpu()
        if x.ndim == 4:
            x = x[0]
        plt.subplot(1, len(images), idx)
        if x.shape[0] == 1:
            plt.imshow(x[0].clamp(0, 1), cmap="magma")
        else:
            plt.imshow(x.clamp(0, 1).permute(1, 2, 0))
        plt.title(title)
        plt.axis("off")
    plt.tight_layout()
    if save_path is not None:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=140, bbox_inches="tight")
    plt.show()

def make_dataset(split="train", crop_size=None, random_crop=False):
    return REVIDESequenceDataset(
        cfg.root_for_split(split),
        split=split,
        seq_len=cfg.seq_len,
        crop_size=crop_size or cfg.crop_size,
        random_crop=random_crop,
        extensions=cfg.image_extensions,
        synthetic_if_empty=True,
        train_mode=cfg.train_mode,
    )

def get_sample_batch(split="train", crop_size=None):
    dataset = make_dataset(split=split, crop_size=crop_size, random_crop=False)
    sample = dataset[0]
    batch = {k: (v.unsqueeze(0).to(DEVICE) if torch.is_tensor(v) else [v]) for k, v in sample.items()}
    return dataset, sample, batch

def run_temporal_debug(batch, use_raft=False):
    frames = batch["frames"].to(DEVICE)
    raft_model = load_raft(DEVICE, cfg.freeze_raft) if use_raft and torch.cuda.is_available() else None
    warped_refs, flows = compute_warped_references_batch(frames, raft_model)
    current = frames[:, -1]
    temporal_memory = TemporalMemoryModule(hidden_dim=64).to(DEVICE)
    transformer = TemporalRetrievalTransformer(
        memory_dim=64,
        token_dim=cfg.transformer_token_dim,
        num_layers=cfg.transformer_num_layers,
        num_heads=cfg.transformer_num_heads,
        pool_size=cfg.transformer_pool_size,
        max_seq_len=cfg.seq_len,
    ).to(DEVICE)
    reference_selector = ReferenceSelectionModule(num_references=cfg.seq_len - 1).to(DEVICE)
    adapter = TemporalConditioningAdapter(cross_attention_dim=768).to(DEVICE)
    aligned = torch.cat([warped_refs, current.unsqueeze(1)], dim=1)
    with torch.no_grad():
        memory = temporal_memory(aligned)
        transformer_out = transformer(aligned, memory)
        memory = transformer_out["enhanced_memory"]
        ref = reference_selector(warped_refs, memory, prior_logits=transformer_out["reference_prior_logits"])
        tokens = adapter(memory, ref["reference_feature"])
    return {
        "frames": frames,
        "warped_refs": warped_refs,
        "flows": flows,
        "memory": memory,
        "transformer": transformer_out,
        "reference": ref,
        "conditioning_tokens": tokens,
        "raft_model": raft_model,
    }

print("Repository imports ready. No notebook-local core implementations were defined.")


In [ ]:
# Cell 7: Dataset Inspection
train_sequences = discover_revide_sequences(Path(cfg.train_root), split=None, extensions=cfg.image_extensions)
test_sequences = discover_revide_sequences(Path(cfg.test_root), split=None, extensions=cfg.image_extensions)
print("TRAIN_ROOT:", cfg.train_root, "exists:", Path(cfg.train_root).exists(), "sequences:", len(train_sequences))
print("TEST_ROOT:", cfg.test_root, "exists:", Path(cfg.test_root).exists(), "sequences:", len(test_sequences))

train_dataset, sample, batch = get_sample_batch(split="train")
print("Dataset length:", len(train_dataset), "sequence:", sample["sequence_name"], "mode:", sample["train_mode"])
for key in ["frames", "current_frame", "target_frame", "mask", "corrupted_frame", "warped_references"]:
    value = sample[key]
    print(f"{key}: shape={tuple(value.shape)} dtype={value.dtype} min={float(value.min()):.4f} max={float(value.max()):.4f}")


In [ ]:
# Cell 8: Dataset Visualization
_, sample, batch = get_sample_batch(split="train")
show_tensor_images(
    [sample["frames"][0], sample["frames"][-1], sample["target_frame"], sample["mask"], sample["corrupted_frame"]],
    ["Frame t-9", "Current input", "Target clean", "Mask", "Corrupted/current"],
    figsize=(18, 4),
    save_path=PATHS["visualizations"] / "dataset_visualization.png",
)


In [ ]:
# Cell 9: Flow Visualization
# Default is zero-flow for Restart + Run All speed. Set RUN_RAFT_FLOW_VIS=True for RAFT on GPU.
RUN_RAFT_FLOW_VIS = False
_, sample, batch = get_sample_batch(split="train")
debug = run_temporal_debug(batch, use_raft=RUN_RAFT_FLOW_VIS)
print("frames:", tuple(debug["frames"].shape))
print("warped_refs:", tuple(debug["warped_refs"].shape))
print("flows:", tuple(debug["flows"].shape))
show_tensor_images(
    [debug["frames"][0, 0].cpu(), flow_to_rgb(debug["flows"][0, 0].cpu()), debug["warped_refs"][0, 0].cpu(), debug["frames"][0, -1].cpu()],
    ["Previous frame", "Flow RGB", "Warped reference", "Current frame"],
    figsize=(16, 4),
    save_path=PATHS["visualizations"] / "flow_visualization.png",
)


In [ ]:
# Cell 10: Mask Visualization
modes = ["rectangle", "ellipse", "blob", "perlin"]
masks = [generate_haze_mask(cfg.image_size, cfg.image_size, mode) for mode in modes]
show_tensor_images(masks, modes, figsize=(14, 4), save_path=PATHS["visualizations"] / "mask_visualization.png")


In [ ]:
# Cell 11: Debug Forward Pass
# Calls repository script. Default skips RAFT and Stable Diffusion for Restart + Run All compatibility.
RUN_FULL_DEBUG_FORWARD = False
cmd = [sys.executable, "debug_forward_pass.py"]
if not RUN_FULL_DEBUG_FORWARD:
    cmd += ["--skip-diffusion", "--no-raft"]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd, cwd=str(REPO_ROOT), text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("debug_forward_pass.py failed")
report_path = Path(cfg.project_root) / "debug_outputs" / "debug_report.json"
if not report_path.exists():
    report_path = REPO_ROOT / "debug_outputs" / "debug_report.json"
if report_path.exists():
    print("Debug report:")
    print(report_path.read_text(encoding="utf-8"))
else:
    print("Debug report was not found; check debug script output above.")


In [ ]:
# Cell 12: Training Configuration
# Edit these controls. Run-all leaves heavy training disabled.
cfg.train_mode = "reconstruct"  # "reconstruct" or "dehaze"
cfg.image_size = 256
cfg.crop_size = 256
cfg.seq_len = 10
cfg.batch_size = 1
cfg.num_workers = 2
cfg.mixed_precision = "fp16"
cfg.use_raft_alignment = True
cfg.use_temporal_transformer = True
cfg.enable_lora = False
cfg.enable_torch_compile = False
cfg.max_train_steps = 1000
cfg.num_epochs = 1
cfg.resume_from_checkpoint = ""

RUN_TRAINING = False
print(json.dumps(cfg.to_dict(), indent=2, default=str))
print("RUN_TRAINING:", RUN_TRAINING)


In [ ]:
# Cell 13: Training Launch
if RUN_TRAINING:
    training_result = train_trdn(cfg)
    print("Training result:", training_result)
else:
    print("Training skipped for Restart + Run All. Set RUN_TRAINING=True in Cell 12 to launch.")
    print("Checkpoints:", PATHS["checkpoints"])
    print("Logs:", PATHS["logs"])


In [ ]:
# Cell 14: Validation
# Heavy validation is disabled by default. It uses src.validate.validate_trdn when enabled.
RUN_VALIDATION = False
VALIDATION_CHECKPOINT = cfg.resume_from_checkpoint
if RUN_VALIDATION:
    diffusion = load_diffusion_backbone(cfg, DEVICE)
    temporal_memory, temporal_transformer, reference_selector, conditioning_adapter = build_temporal_modules(
        cfg, diffusion["unet"].config.cross_attention_dim, DEVICE
    )
    val_loader = DataLoader(make_dataset(split="test", random_crop=False), batch_size=1, shuffle=False, num_workers=0)
    loss_bundle = LossBundle(device=DEVICE)
    metrics = validate_trdn(
        val_loader,
        diffusion,
        temporal_memory,
        temporal_transformer,
        reference_selector,
        conditioning_adapter,
        loss_bundle,
        DEVICE,
        raft_model=None,
        max_batches=4,
        num_steps=min(10, cfg.num_inference_steps),
    )
    print({key: value for key, value in metrics.items() if isinstance(value, float)})
else:
    print("Validation skipped. Set RUN_VALIDATION=True after training or loading a checkpoint.")


In [ ]:
# Cell 15: Inference
# Heavy inference is disabled by default. It uses src.inference.run_inference_on_index when enabled.
RUN_INFERENCE = False
INFERENCE_CHECKPOINT = cfg.resume_from_checkpoint
if RUN_INFERENCE:
    output = run_inference_on_index(cfg, index=0, checkpoint_path=INFERENCE_CHECKPOINT)
    print("Saved prediction:", output["save_path"])
else:
    print("Inference skipped. Set RUN_INFERENCE=True and provide INFERENCE_CHECKPOINT if needed.")


In [ ]:
# Cell 16: Attention Visualization
# Visualizes transformer token-importance maps returned by src.temporal_transformer.
_, sample, batch = get_sample_batch(split="train")
debug = run_temporal_debug(batch, use_raft=False)
importance = debug["transformer"]["token_importance"][0]
current_importance = importance[-1:].cpu()
current_importance = (current_importance - current_importance.min()) / (current_importance.max() - current_importance.min() + 1e-8)
show_tensor_images(
    [current_importance],
    ["Current-frame transformer token importance"],
    figsize=(5, 5),
    save_path=PATHS["visualizations"] / "attention_visualization.png",
)
print("transformer tokens:", tuple(debug["transformer"]["tokens"].shape))
print("conditioning tokens:", tuple(debug["conditioning_tokens"].shape))


In [ ]:
# Cell 17: Reference Weight Visualization
_, sample, batch = get_sample_batch(split="train")
debug = run_temporal_debug(batch, use_raft=False)
weights = debug["reference"]["weights"][0]
num_show = min(5, weights.shape[0])
show_tensor_images(
    [weights[i:i+1].cpu() for i in range(num_show)] + [debug["reference"]["weighted_reference"][0].cpu()],
    [f"Ref {i} weight" for i in range(num_show)] + ["Weighted reference"],
    figsize=(18, 4),
    save_path=PATHS["visualizations"] / "reference_weight_visualization.png",
)
print("reference weights:", tuple(debug["reference"]["weights"].shape))


In [ ]:
# Cell 18: Checkpoint Management
checkpoint_dir = PATHS["checkpoints"]
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoints = sorted([p for p in checkpoint_dir.iterdir() if p.is_dir()])
print("Checkpoint directory:", checkpoint_dir)
if checkpoints:
    print("Available checkpoints:")
    for p in checkpoints[-10:]:
        print(" -", p)
    cfg.resume_from_checkpoint = str(checkpoints[-1])
    print("Default resume_from_checkpoint set to:", cfg.resume_from_checkpoint)
else:
    print("No checkpoints found yet.")
print("TensorBoard logs:", PATHS["logs"])
print("Colab TensorBoard command: %load_ext tensorboard ; %tensorboard --logdir", PATHS["logs"])
